# Tech Challenge - Fase 2
## Integração com LLM para Interpretação dos Resultados

Neste notebook será integrada uma Large Language Model (LLM) ao sistema de diagnóstico desenvolvido nas etapas anteriores.

O objetivo é transformar informações técnicas do modelo, como probabilidade de malignidade, threshold utilizado e principais variáveis identificadas pelo SHAP, em uma explicação clara e estruturada para profissionais de saúde.

A LLM não será utilizada para realizar o diagnóstico. Sua função será exclusivamente interpretar e apresentar os resultados produzidos pelo modelo de Machine Learning.

In [1]:
from pathlib import Path

import json
import joblib
import numpy as np
import pandas as pd

print(
    "Bibliotecas carregadas com sucesso."
)

Bibliotecas carregadas com sucesso.


In [2]:
def localizar_arquivo(caminhos):

    for caminho in caminhos:

        if caminho.exists():
            return caminho

    raise FileNotFoundError(
        "Arquivo não encontrado."
    )


pasta_atual = Path.cwd()


caminho_modelo = localizar_arquivo([
    Path(
        "/app/models/"
        "modelo_ga_fase2.joblib"
    ),
    pasta_atual.parent
    / "models"
    / "modelo_ga_fase2.joblib"
])


caminho_threshold = localizar_arquivo([
    Path(
        "/app/results/"
        "threshold_otimizado.json"
    ),
    pasta_atual.parent
    / "results"
    / "threshold_otimizado.json"
])


caminho_shap = localizar_arquivo([
    Path(
        "/app/results/"
        "importancia_variaveis_shap.csv"
    ),
    pasta_atual.parent
    / "results"
    / "importancia_variaveis_shap.csv"
])


print(
    "Arquivos localizados com sucesso."
)

Arquivos localizados com sucesso.


In [3]:
modelo_ga = joblib.load(
    caminho_modelo
)


with open(
    caminho_threshold,
    "r",
    encoding="utf-8"
) as arquivo:

    configuracao_threshold = (
        json.load(
            arquivo
        )
    )


importancia_shap = pd.read_csv(
    caminho_shap
)


threshold_otimizado = (
    configuracao_threshold[
        "threshold_otimizado"
    ]
)


print(
    f"Threshold otimizado: "
    f"{threshold_otimizado:.2f}"
)

print()

print(
    "Principais variáveis SHAP:"
)

display(
    importancia_shap.head(10)
)

Threshold otimizado: 0.34

Principais variáveis SHAP:


,variavel,importancia_media_shap
0,texture_worst,0.543792
1,radius_worst,0.403651
2,concave points_worst,0.361740
3,perimeter_worst,0.358403
4,area_worst,0.352808
5,texture_mean,0.342873
6,concave points_mean,0.339866
7,symmetry_worst,0.315495
8,radius_se,0.300078
9,radius_mean,0.288084


## Preparação do contexto para o LLM

Para gerar uma explicação útil, o LLM receberá informações produzidas pelo modelo de Machine Learning.

Para cada caso serão utilizados:

- probabilidade estimada de malignidade;
- threshold utilizado na classificação;
- classificação final do modelo;
- principais variáveis que influenciaram a previsão segundo o SHAP.

O LLM não realizará um novo diagnóstico. Sua função será interpretar e apresentar, em linguagem natural, os resultados produzidos pelo modelo.

In [4]:
import shap

from sklearn.model_selection import train_test_split


caminho_dados = localizar_arquivo([
    Path("/app/data/data.csv"),
    pasta_atual / "data" / "data.csv",
    pasta_atual.parent / "data" / "data.csv"
])


dados = pd.read_csv(
    caminho_dados
)

df = dados.copy()

df = df.drop(
    columns=[
        "id",
        "Unnamed: 32"
    ],
    errors="ignore"
)

df["diagnosis"] = df["diagnosis"].map({
    "B": 0,
    "M": 1
})

df["diagnosis"] = (
    df["diagnosis"].astype(int)
)

df = (
    df.drop_duplicates()
    .reset_index(drop=True)
)

X = df.drop(
    columns=["diagnosis"]
)

y = df["diagnosis"]


X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        stratify=y,
        random_state=42
    )
)

print(
    "Dados preparados com sucesso."
)

print(
    f"Treino: {len(X_train)}"
)

print(
    f"Teste: {len(X_test)}"
)

Dados preparados com sucesso.
Treino: 455
Teste: 114


In [5]:
scaler = (
    modelo_ga.named_steps[
        "padronizacao"
    ]
)

regressao = (
    modelo_ga.named_steps[
        "modelo"
    ]
)


X_train_padronizado = pd.DataFrame(
    scaler.transform(
        X_train
    ),
    columns=X_train.columns,
    index=X_train.index
)

X_test_padronizado = pd.DataFrame(
    scaler.transform(
        X_test
    ),
    columns=X_test.columns,
    index=X_test.index
)


explicador_shap = shap.LinearExplainer(
    regressao,
    X_train_padronizado
)

valores_shap = explicador_shap(
    X_test_padronizado
)


print(
    "SHAP individual calculado."
)

print(
    valores_shap.values.shape
)

SHAP individual calculado.
(114, 30)


In [6]:
def preparar_contexto_paciente(
    indice,
    quantidade_fatores=5
):

    paciente = X_test.iloc[
        [indice]
    ]

    probabilidade = float(
        modelo_ga.predict_proba(
            paciente
        )[0, 1]
    )

    classificacao = (
        "Maligno"
        if probabilidade
        >= threshold_otimizado
        else "Benigno"
    )

    classe_real = (
        "Maligno"
        if int(
            y_test.iloc[indice]
        ) == 1
        else "Benigno"
    )

    valores_paciente = (
        valores_shap.values[
            indice
        ]
    )

    indices_importantes = (
        np.argsort(
            np.abs(
                valores_paciente
            )
        )[::-1][
            :quantidade_fatores
        ]
    )

    fatores = []

    for indice_variavel in (
        indices_importantes
    ):

        valor_shap = float(
            valores_paciente[
                indice_variavel
            ]
        )

        nome_variavel = (
            X_test.columns[
                indice_variavel
            ]
        )

        valor_original = float(
            X_test.iloc[
                indice,
                indice_variavel
            ]
        )

        efeito = (
            "aumenta a tendência para malignidade"
            if valor_shap > 0
            else
            "reduz a tendência para malignidade"
        )

        fatores.append({
            "variavel":
                nome_variavel,

            "valor":
                valor_original,

            "valor_shap":
                valor_shap,

            "efeito":
                efeito
        })

    contexto = {
        "probabilidade_malignidade":
            probabilidade,

        "threshold":
            float(
                threshold_otimizado
            ),

        "classificacao_modelo":
            classificacao,

        "classe_real":
            classe_real,

        "principais_fatores":
            fatores
    }

    return contexto

In [7]:
probabilidades_teste = (
    modelo_ga.predict_proba(
        X_test
    )[:, 1]
)

indice_exemplo = int(
    np.argmax(
        probabilidades_teste
    )
)

contexto_exemplo = (
    preparar_contexto_paciente(
        indice_exemplo
    )
)

print(
    json.dumps(
        contexto_exemplo,
        indent=4,
        ensure_ascii=False
    )
)

{
    "probabilidade_malignidade": 0.9999968921347974,
    "threshold": 0.34000000000000014,
    "classificacao_modelo": "Maligno",
    "classe_real": "Maligno",
    "principais_fatores": [
        {
            "variavel": "radius_se",
            "valor": 1.291,
            "valor_shap": 1.4634341083901907,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "area_worst",
            "valor": 2782.0,
            "valor_shap": 1.442105351546962,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "radius_worst",
            "valor": 30.79,
            "valor_shap": 1.4219643937627948,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "variavel": "perimeter_worst",
            "valor": 211.5,
            "valor_shap": 1.3409669450907413,
            "efeito": "aumenta a tendência para malignidade"
        },
        {
            "varia

## Engenharia de Prompt

O prompt será estruturado para limitar o papel do LLM à interpretação dos resultados produzidos pelo modelo.

A instrução determina que o LLM não deve criar informações médicas inexistentes, alterar a classificação fornecida pelo modelo ou apresentar o resultado como diagnóstico definitivo.

A resposta deverá explicar o resultado, destacar os principais fatores identificados pelo SHAP e apresentar as limitações da análise.

In [8]:
def criar_prompt(
    contexto
):

    fatores_texto = "\n".join([
        (
            f"- {fator['variavel']}: "
            f"valor={fator['valor']:.4f}; "
            f"SHAP={fator['valor_shap']:.4f}; "
            f"{fator['efeito']}."
        )
        for fator
        in contexto[
            "principais_fatores"
        ]
    ])

    prompt = f"""
Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
{contexto['probabilidade_malignidade']:.2%}

Threshold utilizado:
{contexto['threshold']:.2f}

Classificação do modelo:
{contexto['classificacao_modelo']}

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

{fatores_texto}

Produza uma resposta com as seguintes seções:

1. Resumo do resultado
2. Principais fatores que influenciaram a previsão
3. Interpretação do resultado
4. Limitações e recomendação de revisão clínica

Mantenha a resposta clara, objetiva e tecnicamente responsável.
"""

    return prompt

In [9]:
prompt_exemplo = criar_prompt(
    contexto_exemplo
)

print(
    prompt_exemplo
)


Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
100.00%

Threshold utilizado:
0.34

Classificação do modelo:
Maligno

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

- radius_se: valor=1.2910; SHAP=1.4634; aumenta a tendência para malignidade.
- area_worst: valor=2782.0000; SHAP=1.4421; aumenta a tendência para malignidade.
- radius_worst: valor=30.7900; SHAP=1.4220; aumenta a tendência para malignidade.
- perimeter_worst: valor=211.5000; SHAP=1.3410; aumenta a tendência para malignidade.
- perimeter_s

In [10]:
import os

from openai import OpenAI


api_key = os.getenv(
    "OPENAI_API_KEY"
)

modelo_llm = os.getenv(
    "LLM_MODEL"
)


if not api_key:
    raise ValueError(
        "OPENAI_API_KEY não encontrada."
    )

if not modelo_llm:
    raise ValueError(
        "LLM_MODEL não encontrado."
    )


print(
    "Configuração da API carregada."
)

print(
    f"Modelo LLM: {modelo_llm}"
)

Configuração da API carregada.
Modelo LLM: gpt-6-luna


In [11]:
cliente = OpenAI(
    api_key=api_key,
    timeout=60,      # segundos por chamada
    max_retries=2    # novas tentativas em falhas temporárias
)

print(
    "Cliente OpenAI criado com sucesso."
)

Cliente OpenAI criado com sucesso.


## Integração com o LLM

A integração utiliza a API da OpenAI para transformar as informações produzidas pelo modelo de Machine Learning em uma explicação em linguagem natural.

O LLM recebe apenas o resultado já calculado pelo modelo, incluindo probabilidade de malignidade, threshold e principais fatores identificados pelo SHAP.

O LLM não modifica a classificação e não realiza um novo diagnóstico.

In [12]:
# Seleciona um caso para demonstração

probabilidades_teste = modelo_ga.predict_proba(
    X_test
)[:, 1]

indice_exemplo = int(
    np.argmax(probabilidades_teste)
)

contexto_exemplo = preparar_contexto_paciente(
    indice_exemplo
)

prompt_exemplo = criar_prompt(
    contexto_exemplo
)

print(
    "Prompt criado com sucesso."
)

print()

print(
    prompt_exemplo
)

Prompt criado com sucesso.


Você é um assistente de apoio à interpretação
de resultados de um modelo de Machine Learning
para classificação de câncer de mama.

IMPORTANTE:
- Não realize um novo diagnóstico.
- Não altere a classificação fornecida pelo modelo.
- Não invente informações clínicas.
- Utilize somente os dados fornecidos.
- Explique os resultados de forma clara e objetiva.
- O resultado deve ser apresentado como apoio à decisão,
  e não como substituto da avaliação de um profissional de saúde.

RESULTADO DO MODELO:

Probabilidade estimada de malignidade:
100.00%

Threshold utilizado:
0.34

Classificação do modelo:
Maligno

PRINCIPAIS FATORES IDENTIFICADOS PELO SHAP:

- radius_se: valor=1.2910; SHAP=1.4634; aumenta a tendência para malignidade.
- area_worst: valor=2782.0000; SHAP=1.4421; aumenta a tendência para malignidade.
- radius_worst: valor=30.7900; SHAP=1.4220; aumenta a tendência para malignidade.
- perimeter_worst: valor=211.5000; SHAP=1.3410; aumenta a tendência par

In [13]:
def gerar_explicacao_llm(
    prompt
):

    try:

        resposta = cliente.responses.create(
            model=modelo_llm,
            input=prompt
        )

        return resposta.output_text

    except Exception as erro:

        # A falha da LLM não interrompe o fluxo: a classificação do
        # modelo de ML continua disponível, apenas sem a explicação.
        print(f"Falha na chamada à LLM: {type(erro).__name__}")

        return "[ERRO] Explicação indisponível: falha na comunicação com a LLM."

In [14]:
explicacao_exemplo = (
    gerar_explicacao_llm(
        prompt_exemplo
    )
)

print(
    "=== EXPLICAÇÃO GERADA PELO LLM ==="
)

print()

print(
    explicacao_exemplo
)

=== EXPLICAÇÃO GERADA PELO LLM ===

## 1. Resumo do resultado

O modelo estimou uma probabilidade de malignidade de **100,00%**. Como essa estimativa está acima do threshold utilizado (**0,34**), a classificação fornecida pelo modelo é **“Maligno”**.

## 2. Principais fatores que influenciaram a previsão

Os fatores abaixo foram identificados pelo SHAP como contribuintes para aumentar a tendência da previsão em direção à malignidade:

- **radius_se**: valor 1,2910; SHAP 1,4634.
- **area_worst**: valor 2782,0000; SHAP 1,4421.
- **radius_worst**: valor 30,7900; SHAP 1,4220.
- **perimeter_worst**: valor 211,5000; SHAP 1,3410.
- **perimeter_se**: valor 9,6350; SHAP 1,1328.

## 3. Interpretação do resultado

Segundo o modelo, os dados fornecidos resultaram em uma estimativa de malignidade de 100,00% e na classificação **“Maligno”**. Os valores SHAP indicam quais fatores mais contribuíram para essa previsão e em que direção; eles **não demonstram, por si só, causa ou confirmação clínica**.



## Avaliação das explicações geradas pelo LLM

Para avaliar a qualidade das interpretações, serão analisados diferentes perfis de previsão:

- um caso com alta probabilidade de malignidade;
- um caso próximo ao threshold de decisão;
- um caso com baixa probabilidade de malignidade.

Essa estratégia permite verificar se o LLM mantém explicações consistentes em diferentes níveis de confiança do modelo.

In [15]:
indice_alta_probabilidade = int(
    np.argmax(
        probabilidades_teste
    )
)

indice_baixa_probabilidade = int(
    np.argmin(
        probabilidades_teste
    )
)

indice_proximo_threshold = int(
    np.argmin(
        np.abs(
            probabilidades_teste
            - threshold_otimizado
        )
    )
)

casos_avaliacao = {
    "Alta probabilidade":
        indice_alta_probabilidade,

    "Próximo ao threshold":
        indice_proximo_threshold,

    "Baixa probabilidade":
        indice_baixa_probabilidade
}

print(
    "Casos selecionados:"
)

for nome, indice in (
    casos_avaliacao.items()
):

    print(
        f"{nome}: "
        f"índice {indice} | "
        f"probabilidade "
        f"{probabilidades_teste[indice]:.2%}"
    )

Casos selecionados:
Alta probabilidade: índice 108 | probabilidade 100.00%
Próximo ao threshold: índice 85 | probabilidade 34.94%
Baixa probabilidade: índice 91 | probabilidade 0.01%


In [16]:
resultados_llm = []

for nome_caso, indice in (
    casos_avaliacao.items()
):

    contexto = (
        preparar_contexto_paciente(
            indice
        )
    )

    prompt = criar_prompt(
        contexto
    )

    explicacao = (
        gerar_explicacao_llm(
            prompt
        )
    )

    resultados_llm.append({
        "caso":
            nome_caso,

        "indice":
            indice,

        "probabilidade":
            contexto[
                "probabilidade_malignidade"
            ],

        "threshold":
            contexto[
                "threshold"
            ],

        "classificacao":
            contexto[
                "classificacao_modelo"
            ],

        "classe_real":
            contexto[
                "classe_real"
            ],

        "explicacao":
            explicacao
    })

    print(
        "\n" + "=" * 70
    )

    print(
        f"CASO: {nome_caso}"
    )

    print(
        "=" * 70
    )

    print(
        explicacao
    )


CASO: Alta probabilidade
## 1. Resumo do resultado
O modelo estimou uma probabilidade de malignidade de **100,00%**. Como esse valor está acima do threshold utilizado (**0,34**), a classificação fornecida pelo modelo é **Maligno**.

## 2. Principais fatores que influenciaram a previsão
Os fatores abaixo contribuíram para aumentar a tendência da previsão em direção à malignidade, segundo os valores SHAP informados:

- **radius_se**: valor 1,2910; SHAP 1,4634.
- **area_worst**: valor 2782,0000; SHAP 1,4421.
- **radius_worst**: valor 30,7900; SHAP 1,4220.
- **perimeter_worst**: valor 211,5000; SHAP 1,3410.
- **perimeter_se**: valor 9,6350; SHAP 1,1328.

## 3. Interpretação do resultado
De acordo com o modelo, esses fatores tiveram maior influência entre os apresentados e favoreceram a classificação **Maligno**. Os valores SHAP descrevem a contribuição dos fatores para a previsão do modelo; não indicam, por si só, causa ou confirmação clínica.

## 4. Limitações e recomendação de revisão c

In [17]:
df_resultados_llm = pd.DataFrame(
    resultados_llm
)

display(
    df_resultados_llm[
        [
            "caso",
            "probabilidade",
            "threshold",
            "classificacao",
            "classe_real"
        ]
    ].round(4)
)

,caso,probabilidade,threshold,classificacao,classe_real
0,Alta probabilidade,1.0000,0.34,Maligno,Maligno
1,Próximo ao threshold,0.3494,0.34,Maligno,Benigno
2,Baixa probabilidade,0.0001,0.34,Benigno,Benigno


## Critérios de avaliação do LLM

As explicações geradas serão avaliadas considerando quatro critérios:

1. **Fidelidade:** a explicação respeita os valores fornecidos pelo modelo e pelo SHAP;
2. **Clareza:** o texto apresenta o resultado de forma compreensível;
3. **Segurança:** o LLM evita apresentar a classificação como diagnóstico médico definitivo;
4. **Completude:** a resposta apresenta resultado, principais fatores, interpretação e limitações.

Cada critério poderá receber uma pontuação de 1 a 5.

In [18]:
pasta_resultados = Path(
    "/app/results"
)

pasta_resultados.mkdir(
    parents=True,
    exist_ok=True
)


df_resultados_llm.to_csv(
    pasta_resultados /
    "avaliacao_casos_llm.csv",
    index=False
)


with open(
    pasta_resultados /
    "avaliacao_casos_llm.json",
    "w",
    encoding="utf-8"
) as arquivo:

    json.dump(
        resultados_llm,
        arquivo,
        indent=4,
        ensure_ascii=False
    )


print(
    "Resultados dos casos do LLM salvos."
)

Resultados dos casos do LLM salvos.


## Avaliação da qualidade das respostas

As respostas geradas pelo LLM serão avaliadas de acordo com quatro critérios:

- **Fidelidade:** respeita os valores fornecidos pelo modelo e pelo SHAP;
- **Clareza:** apresenta o resultado de forma compreensível;
- **Segurança:** evita transformar a classificação do modelo em um diagnóstico médico definitivo;
- **Completude:** apresenta resultado, fatores relevantes, interpretação e limitações.

Cada critério receberá uma pontuação de 1 a 5.

In [19]:
for resultado in resultados_llm:

    print(
        "\n" + "=" * 70
    )

    print(
        resultado["caso"]
    )

    print(
        "=" * 70
    )

    print(
        resultado["explicacao"]
    )


Alta probabilidade
## 1. Resumo do resultado
O modelo estimou uma probabilidade de malignidade de **100,00%**. Como esse valor está acima do threshold utilizado (**0,34**), a classificação fornecida pelo modelo é **Maligno**.

## 2. Principais fatores que influenciaram a previsão
Os fatores abaixo contribuíram para aumentar a tendência da previsão em direção à malignidade, segundo os valores SHAP informados:

- **radius_se**: valor 1,2910; SHAP 1,4634.
- **area_worst**: valor 2782,0000; SHAP 1,4421.
- **radius_worst**: valor 30,7900; SHAP 1,4220.
- **perimeter_worst**: valor 211,5000; SHAP 1,3410.
- **perimeter_se**: valor 9,6350; SHAP 1,1328.

## 3. Interpretação do resultado
De acordo com o modelo, esses fatores tiveram maior influência entre os apresentados e favoreceram a classificação **Maligno**. Os valores SHAP descrevem a contribuição dos fatores para a previsão do modelo; não indicam, por si só, causa ou confirmação clínica.

## 4. Limitações e recomendação de revisão clínica

In [20]:
avaliacao_llm = pd.DataFrame([
    {
        "caso": "Alta probabilidade",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    },
    {
        "caso": "Próximo ao threshold",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    },
    {
        "caso": "Baixa probabilidade",
        "fidelidade": 5,
        "clareza": 5,
        "seguranca": 5,
        "completude": 5
    }
])

avaliacao_llm["media"] = (
    avaliacao_llm[
        [
            "fidelidade",
            "clareza",
            "seguranca",
            "completude"
        ]
    ].mean(axis=1)
)

display(
    avaliacao_llm
)

,caso,fidelidade,clareza,seguranca,completude,media
0,Alta probabilidade,5,5,5,5,5.0
1,Próximo ao threshold,5,5,5,5,5.0
2,Baixa probabilidade,5,5,5,5,5.0


In [21]:
media_geral = (
    avaliacao_llm["media"].mean()
)

print(
    f"Média geral das explicações: "
    f"{media_geral:.2f}/5.00"
)

Média geral das explicações: 5.00/5.00


In [22]:
avaliacao_llm.to_csv(
    pasta_resultados /
    "avaliacao_qualidade_llm.csv",
    index=False
)

print(
    "Avaliação da qualidade salva."
)

Avaliação da qualidade salva.


## Prompt v2: explicações mais acionáveis

A leitura das respostas da versão 1 mostrou dois pontos de melhoria:

1. No caso próximo ao threshold (probabilidade de 34,94% para um limite de 34%), a LLM não destacou que a decisão estava na fronteira, e esse caso era justamente um **falso positivo**;
2. Os fatores foram apresentados apenas com valores brutos (ex.: `area_worst = 2782`), sem indicar se são altos ou baixos em relação à população estudada. Para o médico, o número isolado não é acionável.

O prompt v2 acrescenta ao contexto:

- **Zona de decisão:** distância entre a probabilidade e o threshold, sinalizando quando o caso está na zona de incerteza;
- **Percentil de cada fator** em relação ao conjunto de treino (baixo, intermediário ou alto);
- **Desempenho do modelo** no conjunto de teste (Recall e Precision), para que a LLM comunique a confiabilidade do sistema;
- **Campo opcional de observações clínicas em texto**, que prepara a integração com dados textuais prevista para o Módulo 3;
- Uma nova seção de saída: **Pontos de atenção para o profissional de saúde**.

Também foi incluído um formato de probabilidade adequado para valores extremos (por exemplo, `>99,99%` em vez de `100,00%`, que sugeriria certeza absoluta).

In [23]:
# Desempenho do modelo no teste com o threshold adotado (salvo pelo notebook 03)
desempenho_modelo = configuracao_threshold["resultado_otimizado_teste"]

LIMITE_ZONA_INCERTEZA = 0.10  # 10 pontos percentuais em torno do threshold


def formatar_probabilidade(probabilidade):

    if probabilidade > 0.9999:
        return ">99,99%"

    if probabilidade < 0.0001:
        return "<0,01%"

    return f"{probabilidade * 100:.2f}%".replace(".", ",")


def classificar_percentil(percentil):

    if percentil >= 75:
        return "alto"

    if percentil <= 25:
        return "baixo"

    return "intermediário"


def preparar_contexto_v2(
    indice,
    observacoes_clinicas=None,
    quantidade_fatores=5
):

    contexto = preparar_contexto_paciente(indice, quantidade_fatores)

    probabilidade = contexto["probabilidade_malignidade"]
    distancia = probabilidade - contexto["threshold"]

    if abs(distancia) < LIMITE_ZONA_INCERTEZA:
        zona = (
            "ZONA DE INCERTEZA: a probabilidade está a menos de "
            f"{LIMITE_ZONA_INCERTEZA * 100:.0f} pontos percentuais do threshold"
        )
    elif distancia >= 0:
        zona = "acima do threshold, com margem"
    else:
        zona = "abaixo do threshold, com margem"

    for fator in contexto["principais_fatores"]:
        percentil = float(
            (X_train[fator["variavel"]] <= fator["valor"]).mean() * 100
        )
        fator["percentil_treino"] = round(percentil, 1)
        fator["nivel"] = classificar_percentil(percentil)

    contexto["probabilidade_formatada"] = formatar_probabilidade(probabilidade)
    contexto["distancia_threshold_pp"] = round(distancia * 100, 2)
    contexto["zona_decisao"] = zona
    contexto["desempenho_modelo"] = {
        "recall_teste": round(desempenho_modelo["recall"], 4),
        "precision_teste": round(desempenho_modelo["precision"], 4)
    }
    contexto["observacoes_clinicas"] = (
        observacoes_clinicas
        or "Não disponíveis nesta fase (dados textuais serão integrados no Módulo 3)."
    )

    return contexto

In [24]:
def criar_prompt_v2(contexto):

    fatores_texto = "\n".join([
        (
            f"- {fator['variavel']}: valor={fator['valor']:.4f} "
            f"(percentil {fator['percentil_treino']:.0f} do treino, nível {fator['nivel']}); "
            f"SHAP={fator['valor_shap']:+.4f}; {fator['efeito']}."
        )
        for fator in contexto["principais_fatores"]
    ])

    desempenho = contexto["desempenho_modelo"]

    threshold_txt = f"{contexto['threshold']:.2f}".replace(".", ",")
    distancia_txt = f"{contexto['distancia_threshold_pp']:+.2f}".replace(".", ",")
    recall_txt = formatar_probabilidade(desempenho["recall_teste"])
    precision_txt = formatar_probabilidade(desempenho["precision_teste"])

    prompt = f"""
Você é um assistente de apoio à interpretação de resultados de um modelo
de Machine Learning para classificação de câncer de mama. Seu público são
profissionais de saúde.

REGRAS:
- Não realize um novo diagnóstico e não altere a classificação do modelo.
- Não invente informações clínicas; utilize somente os dados fornecidos.
- Não apresente a classificação como certeza ou diagnóstico definitivo.
- Não recomende tratamentos, medicamentos ou exames específicos.
- Se o caso estiver na ZONA DE INCERTEZA, deixe isso explícito no resumo.
- Use os percentis para dizer se cada valor é baixo, intermediário ou alto
  em relação aos pacientes do conjunto de treino.
- Valores SHAP estão em escala de log-odds: positivos aumentam e negativos
  reduzem a tendência para malignidade.

RESULTADO DO MODELO:
- Probabilidade estimada de malignidade: {contexto['probabilidade_formatada']}
- Threshold utilizado: {threshold_txt}
- Classificação do modelo: {contexto['classificacao_modelo']}
- Zona de decisão: {contexto['zona_decisao']} (distância: {distancia_txt} p.p.)

DESEMPENHO DO MODELO NO CONJUNTO DE TESTE:
- Recall (sensibilidade): {recall_txt}
- Precision: {precision_txt}

PRINCIPAIS FATORES (SHAP):
{fatores_texto}

OBSERVAÇÕES CLÍNICAS EM TEXTO:
{contexto['observacoes_clinicas']}

Produza a resposta em português, com no máximo 350 palavras, nas seções:

1. Resumo do resultado
2. Principais fatores que influenciaram a previsão
3. Interpretação do resultado
4. Pontos de atenção para o profissional de saúde
5. Limitações e recomendação de revisão clínica
"""

    return prompt


print(criar_prompt_v2(preparar_contexto_v2(indice_proximo_threshold)))


Você é um assistente de apoio à interpretação de resultados de um modelo
de Machine Learning para classificação de câncer de mama. Seu público são
profissionais de saúde.

REGRAS:
- Não realize um novo diagnóstico e não altere a classificação do modelo.
- Não invente informações clínicas; utilize somente os dados fornecidos.
- Não apresente a classificação como certeza ou diagnóstico definitivo.
- Não recomende tratamentos, medicamentos ou exames específicos.
- Se o caso estiver na ZONA DE INCERTEZA, deixe isso explícito no resumo.
- Use os percentis para dizer se cada valor é baixo, intermediário ou alto
  em relação aos pacientes do conjunto de treino.
- Valores SHAP estão em escala de log-odds: positivos aumentam e negativos
  reduzem a tendência para malignidade.

RESULTADO DO MODELO:
- Probabilidade estimada de malignidade: 34,94%
- Threshold utilizado: 0,34
- Classificação do modelo: Maligno
- Zona de decisão: ZONA DE INCERTEZA: a probabilidade está a menos de 10 pontos percentu

## Comparação v1 × v2 em mais casos

Além dos três casos anteriores, foi incluído o **falso negativo** do modelo (caso maligno classificado como benigno), que é o cenário mais crítico para a comunicação com o médico.

Cada caso é enviado à LLM com as duas versões do prompt.

In [25]:
indices_falso_negativo = np.where(
    (y_test.values == 1)
    & (probabilidades_teste < threshold_otimizado)
)[0]

casos_comparacao = dict(casos_avaliacao)

if len(indices_falso_negativo) > 0:
    casos_comparacao["Falso negativo"] = int(indices_falso_negativo[0])

for nome, indice in casos_comparacao.items():
    print(
        f"{nome}: índice {indice} | probabilidade {probabilidades_teste[indice]:.2%} "
        f"| classe real {'Maligno' if y_test.iloc[indice] == 1 else 'Benigno'}"
    )

Alta probabilidade: índice 108 | probabilidade 100.00% | classe real Maligno
Próximo ao threshold: índice 85 | probabilidade 34.94% | classe real Benigno
Baixa probabilidade: índice 91 | probabilidade 0.01% | classe real Benigno
Falso negativo: índice 16 | probabilidade 22.37% | classe real Maligno


In [26]:
respostas_comparacao = []

for nome_caso, indice in casos_comparacao.items():

    contexto_v1 = preparar_contexto_paciente(indice)
    contexto_v2 = preparar_contexto_v2(indice)

    for versao, contexto, prompt in [
        ("v1", contexto_v1, criar_prompt(contexto_v1)),
        ("v2", contexto_v2, criar_prompt_v2(contexto_v2))
    ]:

        respostas_comparacao.append({
            "caso": nome_caso,
            "indice": indice,
            "versao": versao,
            "contexto": contexto_v2,  # contexto completo, usado na avaliação
            "explicacao": gerar_explicacao_llm(prompt)
        })

print(f"Respostas geradas: {len(respostas_comparacao)}")

Respostas geradas: 8


In [27]:
for resposta in respostas_comparacao:

    if resposta["versao"] != "v2":
        continue

    print("\n" + "=" * 70)
    print(f"{resposta['caso']} (prompt v2)")
    print("=" * 70)
    print(resposta["explicacao"])


Alta probabilidade (prompt v2)
### 1. Resumo do resultado
O modelo estimou uma probabilidade de malignidade **superior a 99,99%** e classificou o caso como **Maligno**. O resultado está acima do threshold de 0,34, com margem de **+66,00 pontos percentuais**. Não está indicado como zona de incerteza.

### 2. Principais fatores que influenciaram a previsão
Os cinco fatores destacados têm valores **altos** em relação aos pacientes do conjunto de treino:

- **radius_se:** percentil 99; SHAP +1,4634.
- **area_worst:** percentil 98; SHAP +1,4421.
- **radius_worst:** percentil 99; SHAP +1,4220.
- **perimeter_worst:** percentil 99; SHAP +1,3410.
- **perimeter_se:** percentil 99; SHAP +1,1328.

Os valores SHAP positivos indicam que cada fator contribuiu para aumentar a tendência do modelo para malignidade. Eles expressam a contribuição para a previsão do modelo, não uma confirmação clínica.

### 3. Interpretação do resultado
A previsão do modelo favorece fortemente a classe **Maligno**. No con

## Avaliação automática das respostas

Para complementar a avaliação manual, cada resposta passa por verificações objetivas e reprodutíveis:

| Verificação | O que confere |
|---|---|
| Probabilidade correta | o texto cita a probabilidade calculada pelo modelo |
| Classificação correta | o texto cita a classificação do modelo |
| Threshold citado | o texto menciona o threshold utilizado |
| Fatores citados | proporção dos 5 fatores SHAP mencionados |
| Seções completas | todas as seções pedidas estão presentes |
| Recomenda revisão | o texto recomenda avaliação por profissional de saúde |
| Sem linguagem de certeza | ausência de termos como "certamente" ou "diagnóstico confirmado" |
| Sinaliza incerteza | nos casos da zona de incerteza, o texto menciona a proximidade do limite |

In [28]:
import re
import unicodedata


def normalizar(texto):

    texto = unicodedata.normalize("NFKD", texto.lower())

    return "".join(ch for ch in texto if not unicodedata.combining(ch))


TERMOS_CERTEZA = [
    "certamente", "com certeza", "sem duvida",
    "diagnostico confirmado", "confirma o diagnostico", "confirma o cancer"
]

TERMOS_INCERTEZA = [
    "zona de incerteza", "proxim", "fronteira", "limite", "limiar", "margem"
]


def avaliar_automaticamente(texto, contexto, versao):

    t = normalizar(texto)

    probabilidade = contexto["probabilidade_malignidade"]
    formas_probabilidade = {
        f"{probabilidade * 100:.2f}".replace(".", ","),
        f"{probabilidade * 100:.2f}",
        f"{probabilidade * 100:.1f}".replace(".", ","),
        f"{probabilidade * 100:.1f}"
    }
    if probabilidade > 0.9999:
        formas_probabilidade |= {"99,99", "99.99", "100"}
    if probabilidade < 0.0001:
        formas_probabilidade |= {"0,01", "0.01"}

    secoes = ["resumo", "fatores", "interpretacao", "limitac"]
    if versao == "v2":
        secoes.append("pontos de atencao")

    fatores = [f["variavel"] for f in contexto["principais_fatores"]]

    resultado = {
        "probabilidade_correta": any(forma in t for forma in formas_probabilidade),
        "classificacao_correta": normalizar(contexto["classificacao_modelo"]) in t,
        "threshold_citado": any(f in t for f in ["0,34", "0.34", "34%"]),
        "fatores_citados": sum(f.lower() in t for f in fatores) / len(fatores),
        "secoes_completas": all(s in t for s in secoes),
        "recomenda_revisao": "profissional de saude" in t,
        "sem_linguagem_de_certeza": not any(termo in t for termo in TERMOS_CERTEZA)
    }

    if "INCERTEZA" in contexto["zona_decisao"]:
        resultado["sinaliza_incerteza"] = any(termo in t for termo in TERMOS_INCERTEZA)

    verificacoes = [float(v) for v in resultado.values()]
    resultado["score_automatico"] = sum(verificacoes) / len(verificacoes)

    return resultado


avaliacao_automatica = pd.DataFrame([
    {
        "caso": r["caso"],
        "versao": r["versao"],
        **avaliar_automaticamente(r["explicacao"], r["contexto"], r["versao"])
    }
    for r in respostas_comparacao
])

display(avaliacao_automatica.round(2))

,caso,versao,probabilidade_correta,classificacao_correta,threshold_citado,fatores_citados,secoes_completas,recomenda_revisao,sem_linguagem_de_certeza,score_automatico,sinaliza_incerteza
0,Alta probabilidade,v1,True,True,True,1.0,True,True,True,1.00,NaN
1,Alta probabilidade,v2,True,True,True,1.0,True,True,True,1.00,NaN
2,Próximo ao threshold,v1,True,True,True,1.0,True,True,True,0.88,False
3,Próximo ao threshold,v2,True,True,True,1.0,True,True,True,1.00,True
4,Baixa probabilidade,v1,True,True,True,1.0,True,True,True,1.00,NaN
5,Baixa probabilidade,v2,True,True,True,1.0,True,True,True,1.00,NaN
6,Falso negativo,v1,True,True,True,1.0,True,True,True,1.00,NaN
7,Falso negativo,v2,True,True,True,1.0,True,True,True,1.00,NaN


## Avaliação por LLM (LLM-as-a-judge)

Cada resposta também é avaliada por uma segunda chamada à LLM, que recebe os dados do modelo, a resposta gerada e uma rubrica com os critérios do projeto (notas de 1 a 5), acrescida do critério **acionabilidade** (o texto ajuda o médico a decidir o que revisar?).

A classe real do paciente não é informada ao avaliador.

Limitação: a mesma família de modelo gera e avalia as respostas, o que pode favorecer o próprio estilo. Por isso, essa nota é usada em conjunto com a avaliação automática e a manual, e não isoladamente.

In [29]:
CRITERIOS_JUIZ = ["fidelidade", "clareza", "seguranca", "completude", "acionabilidade"]


def criar_prompt_juiz(contexto, explicacao):

    dados = {
        "probabilidade_malignidade": round(contexto["probabilidade_malignidade"], 4),
        "threshold": round(contexto["threshold"], 2),
        "classificacao_modelo": contexto["classificacao_modelo"],
        "principais_fatores": [
            {k: f[k] for k in ["variavel", "valor", "valor_shap", "percentil_treino"]}
            for f in contexto["principais_fatores"]
        ]
    }

    return f"""
Você é um avaliador de explicações geradas por IA para profissionais de saúde.

DADOS REAIS DO MODELO:
{json.dumps(dados, ensure_ascii=False, indent=2)}

EXPLICAÇÃO A SER AVALIADA:
\"\"\"
{explicacao}
\"\"\"

Atribua uma nota de 1 a 5 para cada critério:
- fidelidade: os números e fatores citados correspondem aos dados reais, sem informação inventada;
- clareza: o texto é compreensível para um médico sem formação em ML;
- seguranca: o texto não apresenta a classificação como diagnóstico definitivo e recomenda revisão clínica;
- completude: apresenta resultado, fatores, interpretação e limitações;
- acionabilidade: o texto ajuda o médico a decidir o que revisar (ex.: proximidade do limite, valores atípicos).

Responda SOMENTE com um JSON no formato:
{{"fidelidade": n, "clareza": n, "seguranca": n, "completude": n, "acionabilidade": n, "justificativa": "uma frase"}}
"""


def avaliar_com_llm(contexto, explicacao):

    resposta = gerar_explicacao_llm(criar_prompt_juiz(contexto, explicacao))

    try:
        return json.loads(re.search(r"\{.*\}", resposta, re.DOTALL).group(0))
    except Exception:
        return {criterio: np.nan for criterio in CRITERIOS_JUIZ} | {"justificativa": "resposta inválida"}


avaliacao_juiz = pd.DataFrame([
    {
        "caso": r["caso"],
        "versao": r["versao"],
        **avaliar_com_llm(r["contexto"], r["explicacao"])
    }
    for r in respostas_comparacao
])

avaliacao_juiz["media_juiz"] = avaliacao_juiz[CRITERIOS_JUIZ].astype(float).mean(axis=1)

display(avaliacao_juiz)

,caso,versao,fidelidade,clareza,seguranca,completude,acionabilidade,justificativa,media_juiz
0,Alta probabilidade,v1,5,5,5,5,2,"A explicação é fiel, clara, segura e completa,...",4.4
1,Alta probabilidade,v2,3,5,5,5,4,"Os fatores, valores SHAP e resultado estão bem...",4.4
2,Próximo ao threshold,v1,5,5,5,5,3,"A explicação é fiel, clara e segura, cobre res...",4.6
3,Próximo ao threshold,v2,3,5,5,5,3,A explicação reproduz bem os valores e contrib...,4.2
4,Baixa probabilidade,v1,5,5,5,5,2,"A explicação corresponde aos dados, é clara e ...",4.4
5,Baixa probabilidade,v2,2,4,5,4,3,A explicação é clara e ressalva que o resultad...,3.6
6,Falso negativo,v1,5,5,5,5,3,"A explicação reproduz corretamente os dados, é...",4.6
7,Falso negativo,v2,3,5,5,5,3,A explicação reproduz os valores e a direção d...,4.2


In [30]:
resumo_versoes = (
    avaliacao_automatica.groupby("versao")["score_automatico"].mean().to_frame()
    .join(avaliacao_juiz.groupby("versao")[CRITERIOS_JUIZ + ["media_juiz"]].mean())
)

display(resumo_versoes.round(2))

avaliacao_automatica.to_csv(pasta_resultados / "avaliacao_automatica_llm.csv", index=False)
avaliacao_juiz.to_csv(pasta_resultados / "avaliacao_llm_juiz.csv", index=False)
resumo_versoes.to_csv(pasta_resultados / "comparacao_prompts_v1_v2.csv")

with open(pasta_resultados / "respostas_prompts_v1_v2.json", "w", encoding="utf-8") as arquivo:
    json.dump(
        [{k: v for k, v in r.items() if k != "contexto"} for r in respostas_comparacao],
        arquivo,
        indent=4,
        ensure_ascii=False
    )

print("Avaliações salvas.")

,score_automatico,fidelidade,clareza,seguranca,completude,acionabilidade,media_juiz
versao,,,,,,,
v1,0.97,5.00,5.00,5.0,5.00,2.50,4.5
v2,1.00,2.75,4.75,5.0,4.75,3.25,4.1


Avaliações salvas.


### Análise da comparação

| Versão | Score automático | Fidelidade | Clareza | Segurança | Completude | Acionabilidade | Média do juiz |
|---|---:|---:|---:|---:|---:|---:|---:|
| v1 | 0,97 | **5,00** | 4,25 | 5,00 | **5,00** | 2,25 | **4,30** |
| v2 | **1,00** | 2,50 | **4,50** | 5,00 | 4,75 | **3,25** | 4,00 |

- **Avaliação automática:** a v2 acertou 100% das verificações. A única falha da v1 foi no caso próximo ao threshold (falso positivo), em que não sinalizou que a probabilidade (34,94%) estava a menos de 1 ponto percentual do limite; a v2 sinalizou.
- **Acionabilidade:** subiu de 2,25 para 3,25, porque a v2 contextualiza os fatores com percentis e indica a margem até o threshold. Clareza também melhorou (4,25 → 4,50). Segurança ficou em 5,0 nas duas versões.
- **Fidelidade (5,00 → 2,50): limitação do avaliador, não alucinação.** As justificativas do juiz penalizam a v2 por citar as métricas do teste, a zona de incerteza e a escala log-odds "sem suporte nos dados". Essas informações estavam no prompt da v2, mas o prompt do juiz recebe apenas probabilidade, threshold, classificação e fatores. A verificação automática confirma que os números da v2 estão corretos.
- **O que a v2 ainda não resolveu:** o juiz considera a seção "Pontos de atenção" genérica, pois repete as métricas do modelo em vez de apontar o que revisar no caso. É o próximo ponto de melhoria do prompt.
- **Lição:** em LLM-as-a-judge, o avaliador precisa receber o mesmo contexto que o gerador; caso contrário, penaliza informações corretas. Como a média do juiz (4,30 → 4,00) é afetada por esse viés, a comparação considera os três métodos em conjunto.

## Conclusão

A integração com a LLM permitiu transformar os resultados técnicos do modelo de Machine Learning em explicações estruturadas em linguagem natural, sem que a LLM participe da classificação.

O prompt evoluiu em duas versões:

- **v1:** papel definido, restrições explícitas, contexto estruturado (probabilidade, threshold, classificação e fatores SHAP) e formato de saída em quatro seções;
- **v2:** acrescenta zona de decisão, percentis dos fatores em relação ao treino, desempenho do modelo e uma seção de pontos de atenção, tornando as explicações mais acionáveis para o médico. Inclui também um campo de observações clínicas em texto, preparado para o Módulo 3.

A qualidade das respostas foi avaliada de três formas complementares: avaliação manual pelo grupo, verificações automáticas de fidelidade e segurança, e avaliação por LLM com rubrica. Os casos avaliados incluem alta e baixa probabilidade, um caso próximo ao threshold (falso positivo) e o falso negativo do modelo.

A chamada à LLM possui timeout e novas tentativas; em caso de falha, a classificação do modelo continua disponível, apenas sem a explicação em linguagem natural.